<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/StereoQueerEval_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [113]:
!pip install tensorflow

In [114]:
import pandas as pd

In [115]:
df = pd.read_csv('/content/StereoQueerEval_EN_training.tsv', sep='\t')
df.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g"
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g


In [116]:
df = df.drop(['hate_speech', 'target','StereoQueerEval_id'], axis=1)
df.head()

,yt_title,yt_description,yt_comment,stereotype
0,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes
1,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes
2,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes
3,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes
4,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes


In [117]:
df.isnull().sum()

,0
yt_title,0
yt_description,0
yt_comment,0
stereotype,1


In [118]:
df.dropna(inplace=True)

In [119]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4449 entries, 0 to 4449
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   yt_title        4449 non-null   object
 1   yt_description  4449 non-null   object
 2   yt_comment      4449 non-null   object
 3   stereotype      4449 non-null   object
dtypes: object(4)
memory usage: 173.8+ KB


In [120]:
df.shape

(4449, 4)

In [121]:
import tensorflow as tf
tf.__version__

'2.21.0'

In [122]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dense
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [123]:
X = df.drop('stereotype',axis=1)
y = df['stereotype']

In [124]:
X

,yt_title,yt_description,yt_comment
0,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...
1,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...
2,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall..."
3,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...
4,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him
...,...,...,...
4445,Lesbian magazine Curve 'saved people from lone...,A new documentary called Ahead of the Curve ch...,Somewhere in germany Kolon some guy has taken ...
4446,TURKEY: RIOT POLICE BREAK UP GAY PRIDE PARADE,"To License This Clip, Click Here: [URL]",Respect from Uzbekistan
4447,Argentina’s transgender soccer star,"Meet Mara Gomez, Argentina’s first transgender...",Yeah.. super fair to all those REAL women.
4448,Model comes out as intersex,Model Hanne Gaby Odiele has revealed that she ...,Good for her. Everyone has the right to be wan...


In [125]:
y

,stereotype
0,yes
1,yes
2,yes
3,yes
4,yes
...,...
4445,no
4446,no
4447,yes
4448,no


In [126]:
import nltk
import re
from nltk.corpus import stopwords

In [127]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [128]:
len(X)

4449

#different corpus for different column

In [129]:
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()
corpus_title = []
corpus_description = []
corpus_comment = []

# Process yt_title
for text in X['yt_title']:
    review = re.sub('[^a-zA-Z]', ' ', str(text))
    review = review.lower().split()
    review = [stemmer.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus_title.append(review)

# Process yt_description
for text in X['yt_description']:
    review = re.sub('[^a-zA-Z]', ' ', str(text))
    review = review.lower().split()
    review = [stemmer.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus_description.append(review)

# Process yt_comment
for text in X['yt_comment']:
    review = re.sub('[^a-zA-Z]', ' ', str(text))
    review = review.lower().split()
    review = [stemmer.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus_comment.append(review)

In [130]:
voc_size = 5000

In [131]:
one_hot_rep_comment=[one_hot(word,voc_size)for word in corpus_comment]
one_hot_rep_description=[one_hot(word,voc_size)for word in corpus_description]
one_hot_rep_title=[one_hot(word,voc_size)for word in corpus_title]

In [132]:
print(corpus_comment[1])
print(one_hot_rep_comment[1])

rep mace keep dedic truth polit correct propaganda thank leadership
[289, 189, 3917, 3568, 4433, 3631, 3276, 3347, 3795, 1721]


In [133]:
max_length=30
embedded_doc_comment=pad_sequences(one_hot_rep_comment,padding='post',maxlen=max_length)
embedded_doc_description=pad_sequences(one_hot_rep_description,padding='post',maxlen=max_length)
embedded_doc_title=pad_sequences(one_hot_rep_title,padding='post',maxlen=max_length)

In [134]:
print(embedded_doc_title)
print(embedded_doc_description)
print(embedded_doc_comment)

[[2914 3899 1945 ...    0    0    0]
 [4603  830 1173 ...    0    0    0]
 [1004 2534   23 ...    0    0    0]
 ...
 [4622 2041 4442 ...    0    0    0]
 [3724 4720 3770 ...    0    0    0]
 [3640  406  608 ...    0    0    0]]
[[2914 2015 2556 ...    0    0    0]
 [1152 3473 1173 ...  584 2914 2169]
 [1004 2534  917 ...    0    0    0]
 ...
 [2914 2687 2554 ... 2914 3432 2899]
 [3724 4165  924 ...    0    0    0]
 [3411 3629 3640 ...    0    0    0]]
[[3118 4251  256 ...    0    0    0]
 [ 289  189 3917 ...    0    0    0]
 [2191 4420 1547 ...    0    0    0]
 ...
 [  90 1856 4462 ...    0    0    0]
 [ 464 4334  824 ...    0    0    0]
 [3709 2929  321 ...    0    0    0]]


In [135]:
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=max_length))
model.add(LSTM(150))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [136]:
embeded_doc = pd.DataFrame({
    'yt_title': list(embedded_doc_title),
    'yt_description': list(embedded_doc_description),
    'yt_comment': list(embedded_doc_comment)
})
display(embeded_doc.head())

,yt_title,yt_description,yt_comment
0,"[2914, 3899, 1945, 3863, 4204, 0, 0, 0, 0, 0, ...","[2914, 2015, 2556, 4547, 813, 4747, 1945, 3863...","[3118, 4251, 256, 3461, 3405, 2136, 0, 0, 0, 0..."
1,"[4603, 830, 1173, 3254, 284, 1968, 3130, 4573,...","[1152, 3473, 1173, 3254, 2041, 1968, 2140, 376...","[289, 189, 3917, 3568, 4433, 3631, 3276, 3347,..."
2,"[1004, 2534, 23, 3299, 3936, 0, 0, 0, 0, 0, 0,...","[1004, 2534, 917, 3299, 824, 23, 3936, 2981, 3...","[2191, 4420, 1547, 4584, 1369, 796, 491, 3461,..."
3,"[355, 4880, 1150, 3299, 1456, 2120, 0, 0, 0, 0...","[504, 355, 4880, 1456, 1150, 3899, 4, 813, 147...","[1235, 3121, 4394, 4454, 1768, 1173, 2596, 338..."
4,"[295, 4734, 4232, 3855, 3007, 3299, 1358, 2914...","[824, 774, 3735, 824, 3939, 2170, 4613, 3855, ...","[1896, 2689, 4933, 4234, 0, 0, 0, 0, 0, 0, 0, ..."


In [137]:
y = y.map({'no': 0, 'yes': 1})

In [138]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(embeded_doc,y,test_size=0.2,random_state=45)

In [139]:
X_train.shape,X_test.shape

((3559, 3), (890, 3))

In [140]:
import numpy as np

X_train = np.hstack([
    np.stack(X_train['yt_title'].values),
    np.stack(X_train['yt_description'].values),
    np.stack(X_train['yt_comment'].values)
])

X_test = np.hstack([
    np.stack(X_test['yt_title'].values),
    np.stack(X_test['yt_description'].values),
    np.stack(X_test['yt_comment'].values)
])

In [141]:
epochs=30
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=epochs,batch_size=64)

Epoch 1/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.5881 - loss: 0.6741 - val_accuracy: 0.6281 - val_loss: 0.6548
Epoch 2/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6510 - loss: 0.6429 - val_accuracy: 0.6663 - val_loss: 0.6225
Epoch 3/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7128 - loss: 0.5815 - val_accuracy: 0.7124 - val_loss: 0.5841
Epoch 4/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7710 - loss: 0.4906 - val_accuracy: 0.7337 - val_loss: 0.5630
Epoch 5/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8289 - loss: 0.4012 - val_accuracy: 0.7820 - val_loss: 0.4996
Epoch 6/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8651 - loss: 0.3331 - val_accuracy: 0.8000 - val_loss: 0.4812
Epoch 7/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8896 - loss: 0.2811 - val_accuracy: 0.7910 - val_loss: 0.5468
Epoch 8/30
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9151 - loss: 0.2331 - val_accuracy: 0.8146 - v

In [142]:
y_pred=model.predict(X_test)

28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


In [146]:
import numpy as np
y_pred=np.where(y_pred>0.5,1,0)

In [147]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_pred,y_test))

0.8337078651685393


In [148]:
from sklearn.metrics import classification_report
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.91      0.82      0.86       574
           1       0.73      0.85      0.78       316

    accuracy                           0.83       890
   macro avg       0.82      0.84      0.82       890
weighted avg       0.85      0.83      0.84       890

